In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import json
from pathlib import Path
from transformers import PretrainedConfig, PreTrainedModel, AutoTokenizer

# ==========================================
# 1. CONFIGURATION
# ==========================================
CHECKPOINT_PATH = "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/TrainGen"
OUTPUT_PACKAGE_DIR = "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/ONNXModels/TrainGenONNX"
ONNX_FILENAME = "model.onnx"

# ==========================================
# 2. ARCHITECTURE DEFINITIONS
# ==========================================

class TitansConfig(PretrainedConfig):
    model_type = "titans"
    def __init__(self, vocab_size=30522, hidden_size=256, num_hidden_layers=4, 
                 num_attention_heads=4, intermediate_size=1024, num_persistent_tokens=16, 
                 segment_len=128, num_labels=2, ttt_lr=0.01, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.num_persistent_tokens = num_persistent_tokens
        self.segment_len = segment_len
        self.num_labels = num_labels
        self.ttt_lr = ttt_lr

class NeuralMemoryTTT(nn.Module):
    def __init__(self, hidden_dim, ttt_lr):
        super().__init__()
        self.ttt_lr = ttt_lr

    def forward(self, x, current_M):
        retrieved = torch.bmm(x, current_M) 
        error = retrieved - x
        grad = torch.bmm(x.transpose(1, 2), error)
        new_M = current_M - self.ttt_lr * grad
        return retrieved.squeeze(1), new_M

class ManualExportMultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, batch_first=True):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.batch_first = batch_first 
        self.in_proj_weight = nn.Parameter(torch.empty(3 * embed_dim, embed_dim))
        self.in_proj_bias = nn.Parameter(torch.empty(3 * embed_dim))
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value):
        qkv = F.linear(query, self.in_proj_weight, self.in_proj_bias)
        batch_size, seq_len, _ = qkv.shape
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = (q @ k.transpose(-2, -1)) * self.scale
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_output = (attn_probs @ v).transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)
        return self.out_proj(attn_output)

class ExportFriendlyTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, intermediate_size=1024):
        super().__init__()
        self.self_attn = ManualExportMultiheadAttention(d_model, nhead, batch_first=True)
        self.linear1 = nn.Linear(d_model, intermediate_size)
        self.linear2 = nn.Linear(intermediate_size, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.activation = F.relu 

    # --- FIX: Added **kwargs to catch 'src_mask', 'is_causal', etc. ---
    def forward(self, src, **kwargs):
        src_norm = self.norm1(src)
        attn_output = self.self_attn(src_norm, src_norm, src_norm)
        src = src + attn_output
        src_norm2 = self.norm2(src)
        ff_output = self.linear2(self.activation(self.linear1(src_norm2)))
        src = src + ff_output
        return src

class TitansMACClassifier(PreTrainedModel):
    config_class = TitansConfig
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.persistent_memory = nn.Parameter(torch.randn(config.num_persistent_tokens, config.hidden_size))
        self.neural_memory = NeuralMemoryTTT(config.hidden_size, config.ttt_lr)
        
        custom_layer = ExportFriendlyTransformerLayer(
            d_model=config.hidden_size,
            nhead=config.num_attention_heads,
            intermediate_size=config.intermediate_size
        )
        self.core_attention = nn.TransformerEncoder(custom_layer, num_layers=config.num_hidden_layers)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        
    def forward(self, input_ids, memory_in):
        x = self.embedding(input_ids)
        batch_size, seq_len, hidden_size = x.shape
        M = memory_in
        
        segments = x.split(self.config.segment_len, dim=1)
        final_out = None
        
        for segment in segments:
            summary_in = segment.mean(dim=1).unsqueeze(1)
            context = torch.bmm(summary_in, M)
            persistent_batch = self.persistent_memory.unsqueeze(0).expand(batch_size, -1, -1)
            combined_input = torch.cat([persistent_batch, context, segment], dim=1)
            
            core_out = self.core_attention(combined_input)
            segment_summary = core_out[:, -self.config.segment_len:, :].mean(dim=1).unsqueeze(1)
            
            # Neural Memory Adaptation
            final_out, M = self.neural_memory(segment_summary, M)
            
        logits = self.classifier(final_out)
        return logits, M 

# ==========================================
# 3. EXPORT PIPELINE
# ==========================================

def export_pipeline():
    print(f"--- Exporting Stateful Titans TTT ---")
    Path(OUTPUT_PACKAGE_DIR).mkdir(parents=True, exist_ok=True)
    onnx_path = os.path.join(OUTPUT_PACKAGE_DIR, ONNX_FILENAME)

    print("[1/3] Loading TTT Model...")
    # Load with custom config if necessary
    model = TitansMACClassifier.from_pretrained(CHECKPOINT_PATH, local_files_only=True)
    model.eval().cpu()

    print("[2/3] Tracing Stateful ONNX Graph...")
    dummy_input = torch.randint(0, 30522, (1, 512), dtype=torch.long)
    dummy_memory = torch.eye(model.config.hidden_size).unsqueeze(0) 

    torch.onnx.export(
        model,
        (dummy_input, dummy_memory),
        onnx_path,
        export_params=True,
        opset_version=14, 
        do_constant_folding=True,
        input_names=['input_ids', 'memory_in'],
        output_names=['logits', 'memory_out'],
        dynamic_axes={
            'input_ids': {0: 'batch_size'},
            'memory_in': {0: 'batch_size'},
            'memory_out': {0: 'batch_size'}
        }
    )
    
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)
    tokenizer.save_pretrained(OUTPUT_PACKAGE_DIR)
    print(f"✅ Stateful ONNX exported to: {onnx_path}")

if __name__ == "__main__":
    export_pipeline()

Some weights of the model checkpoint at /home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/TrainGen were not used when initializing TitansMACClassifier: ['neural_memory.memory_cell.bias_hh', 'neural_memory.memory_cell.bias_ih', 'neural_memory.memory_cell.weight_hh', 'neural_memory.memory_cell.weight_ih']
- This IS expected if you are initializing TitansMACClassifier from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TitansMACClassifier from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


--- Exporting Stateful Titans TTT ---
[1/3] Loading TTT Model...
[2/3] Tracing Stateful ONNX Graph...
✅ Stateful ONNX exported to: /home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/ONNXModels/TrainGenONNX/model.onnx


In [56]:
import glob
import numpy as np
import os
import onnxruntime as ort
from tqdm import tqdm
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
from transformers import AutoTokenizer

# CRITICAL: Ensures data is loaded exactly as it was during training
from utils import SegmentFromFile 

# --- CONFIGURATION ---
ONNX_FILE_NAME = "model.onnx"
# Path to your stateful ONNX package (must have memory_in/memory_out ports)
ONNX_MODEL_DIRECTORY = "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/ONNXModels/TrainGenONNX" 
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Val"
TIME_GAP_TEST = 98.0
HIDDEN_SIZE = 256  # Must match your TitansConfig.hidden_size

def evaluate_titans_chunk_learning(model_path, validation_dir, time_gap, device="cuda"):
    """
    Evaluates the Titans model by learning from chunk to chunk.
    This is true Meta-Learning: Chunk N informs Chunk N+1.
    """
    onnx_file_path = os.path.join(model_path, ONNX_FILE_NAME)
    print(f"\n--- Starting Chunk-Based Learning Evaluation: {onnx_file_path} ---")

    # 1. Setup Session & Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device == "cuda" else ['CPUExecutionProvider']
    session = ort.InferenceSession(onnx_file_path, providers=providers)
    
    input_names = [i.name for i in session.get_inputs()]   # Expects ['input_ids', 'memory_in']
    output_names = [o.name for o in session.get_outputs()] # Expects ['logits', 'memory_out']

    # 2. Sequential Data Loading
    csv_files = sorted(glob.glob(os.path.join(validation_dir, "*.csv")))
    
    all_preds = []
    all_labels = []

    # INITIALIZE MEMORY (The 'Learning' State)
    # We define this OUTSIDE the file loop so the model can learn across multiple files/chunks
    current_memory = np.eye(HIDDEN_SIZE, dtype=np.float32)[np.newaxis, :, :]

    for file_path in tqdm(csv_files, desc="Processing Files"):
        filename = os.path.basename(file_path)
        # Load chunks for the specific file
        chunks, labels = SegmentFromFile(validation_dir, filename, time_gap=time_gap)
        all_labels.extend(labels)

        for chunk in chunks:
            # Format: Txx Gxx Txx Gxx...
            text = " ".join([f"T{int(p[0])} G{int(p[1])}" for p in chunk])
            
            inputs = tokenizer(
                text,
                return_tensors="np",
                padding="max_length",
                truncation=True,
                max_length=512
            )

            # --- THE META-LEARNING STEP ---
            # We pass the memory matrix learned from the PREVIOUS chunk
            ort_inputs = {
                'input_ids': inputs['input_ids'].astype(np.int64),
                'memory_in': current_memory
            }

            # Run: Logits + Updated Memory
            outputs = session.run(output_names, ort_inputs)
            
            # Update the memory matrix: The model 'learned' from this chunk
            # The next chunk will use this updated knowledge
            current_memory = outputs[1] 
            
            # Store prediction
            all_preds.append(np.argmax(outputs[0], axis=-1)[0])

    # 3. Calculate Final Metrics
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', zero_division=0)
    
    print("\n=== Titans Chunk-Learning Results ===")
    print(f"Accuracy:  {acc:.4f} (Increased over {len(all_labels)} chunks)")
    print(f"F1 Score:  {f1:.4f}")
    print("=" * 35)

In [57]:
# --- CONFIGURATION ---
ONNX_FILE_NAME = "model.onnx"
# Path to your stateful TTT ONNX model package
ONNX_MODEL_DIRECTORY = "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/ONNXModels/TrainGenONNX" 
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla/Lower Low"
TIME_GAP_TEST = 98.0
HIDDEN_SIZE = 256  # Must match TitansConfig.hidden_size
DEVICE = "cuda"    # "cuda" or "cpu"

In [58]:
evaluate_titans_chunk_learning(
        model_path=ONNX_MODEL_DIRECTORY,
        validation_dir=VALIDATION_DATA_DIRECTORY,
        time_gap=TIME_GAP_TEST
    )


--- Starting Chunk-Based Learning Evaluation: /home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/ONNXModels/TrainGenONNX/model.onnx ---


Processing Files: 100%|██████████| 4/4 [00:13<00:00,  3.40s/it]


=== Titans Chunk-Learning Results ===
Accuracy:  0.3104 (Increased over 2838 chunks)
F1 Score:  0.4732


In [23]:
# --- CONFIGURATION ---
ONNX_FILE_NAME = "model.onnx"
# Path where you saved the ONNX package (must contain model.onnx and tokenizer files)
ONNX_MODEL_DIRECTORY = "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/OnnxModels/TrainSilOnnx" 
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla/Lower Low"
TIME_GAP_TEST = 83.0

In [24]:
want_gpu = True 

evaluate_model_onnx(
    model_path=ONNX_MODEL_DIRECTORY,
    validation_data_directory=VALIDATION_DATA_DIRECTORY,
    time_gap=TIME_GAP_TEST,
    use_gpu=want_gpu
)


--- Starting Evaluation of ONNX Model: /home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/OnnxModels/TrainSilOnnx/model.onnx ---
Loading Tokenizer...
✅ Using GPU (CUDAExecutionProvider)
Model Inputs: ['input_ids']
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla/Lower Low


Processing files: 100%|██████████| 4/4 [00:03<00:00,  1.04it/s]


Running inference...


100%|██████████| 210/210 [00:02<00:00, 83.21it/s]


Calculating metrics...

=== Titans ONNX Results ===
Accuracy:  0.9122
F1 Score:  0.8726
Precision: 0.7929
Recall:    0.9701

=== Confusion Matrix ===
Actual Neg | Pred Neg: 2049 | Pred Pos: 263
Actual Pos | Pred Neg: 31 | Pred Pos: 1007


In [25]:
# --- CONFIGURATION ---
ONNX_FILE_NAME = "model.onnx"
# Path where you saved the ONNX package (must contain model.onnx and tokenizer files)
ONNX_MODEL_DIRECTORY = "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/OnnxModels/TrainKiaOnnx" 
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla/Lower Low"
TIME_GAP_TEST = 83.0

In [26]:
want_gpu = True 

evaluate_model_onnx(
    model_path=ONNX_MODEL_DIRECTORY,
    validation_data_directory=VALIDATION_DATA_DIRECTORY,
    time_gap=TIME_GAP_TEST,
    use_gpu=want_gpu
)


--- Starting Evaluation of ONNX Model: /home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/OnnxModels/TrainKiaOnnx/model.onnx ---
Loading Tokenizer...
✅ Using GPU (CUDAExecutionProvider)
Model Inputs: ['input_ids']
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla/Lower Low


Processing files: 100%|██████████| 4/4 [00:03<00:00,  1.00it/s]


Running inference...


100%|██████████| 210/210 [00:02<00:00, 81.36it/s]



Calculating metrics...

=== Titans ONNX Results ===
Accuracy:  0.9122
F1 Score:  0.8726
Precision: 0.7929
Recall:    0.9701

=== Confusion Matrix ===
Actual Neg | Pred Neg: 2049 | Pred Pos: 263
Actual Pos | Pred Neg: 31 | Pred Pos: 1007


In [30]:
# --- CONFIGURATION ---
ONNX_FILE_NAME = "model.onnx"
# Path where you saved the ONNX package (must contain model.onnx and tokenizer files)
ONNX_MODEL_DIRECTORY = "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/OnnxModels/TrainTeslaOnnx" 
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Kia/Lower Low"
TIME_GAP_TEST = 100.0

In [31]:
want_gpu = True 

evaluate_model_onnx(
    model_path=ONNX_MODEL_DIRECTORY,
    validation_data_directory=VALIDATION_DATA_DIRECTORY,
    time_gap=TIME_GAP_TEST,
    use_gpu=want_gpu
)


--- Starting Evaluation of ONNX Model: /home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/titans/OnnxModels/TrainTeslaOnnx/model.onnx ---
Loading Tokenizer...
✅ Using GPU (CUDAExecutionProvider)
Model Inputs: ['input_ids']
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Kia/Lower Low


Processing files: 100%|██████████| 4/4 [00:06<00:00,  1.52s/it]


Running inference...


100%|██████████| 207/207 [00:02<00:00, 82.94it/s]


Calculating metrics...

=== Titans ONNX Results ===
Accuracy:  0.9601
F1 Score:  0.9367
Precision: 0.9243
Recall:    0.9495

=== Confusion Matrix ===
Actual Neg | Pred Neg: 2202 | Pred Pos: 80
Actual Pos | Pred Neg: 52 | Pred Pos: 977
